In [1]:
import numpy as np

class SoftmaxRegression:
    def __init__(self, lr=0.1, epochs=1000):
        self.lr = lr
        self.epochs = epochs

    def softmax(self, Z):
        Z = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

    def fit(self, X, y):
        n, d = X.shape
        self.classes = np.unique(y)
        K = len(self.classes)

        # Convert class labels to integer indices
        y_idx = np.searchsorted(self.classes, y)
        Y = np.eye(K)[y_idx]

        self.W = np.zeros((d, K))
        self.b = np.zeros(K)
        self.loss_history = []

        for _ in range(self.epochs):
            # 1. Scores and probabilities
            Z = X @ self.W + self.b
            P = self.softmax(Z)

            # 2. Cross-entropy loss
            eps = 1e-15
            loss = -np.mean(
                np.sum(Y * np.log(np.clip(P, eps, 1)), axis=1)
            )
            self.loss_history.append(loss)

            # 3. Gradients
            dW = X.T @ (P - Y) / n
            db = np.mean(P - Y, axis=0)

            # 4. Update parameters
            self.W -= self.lr * dW
            self.b -= self.lr * db

        return self

    def predict_proba(self, X):
        return self.softmax(X @ self.W + self.b)

    def predict(self, X):
        indices = np.argmax(self.predict_proba(X), axis=1)
        return self.classes[indices]


# Three-class example
X = np.array([
    [1, 2], [2, 1], [1, 1],
    [5, 6], [6, 5], [5, 5],
    [9, 1], [8, 2], [9, 2]
], dtype=float)

y = np.array([
    "A", "A", "A",
    "B", "B", "B",
    "C", "C", "C"
])

model = SoftmaxRegression(lr=0.1, epochs=1000)
model.fit(X, y)

print("Probabilities:\n", model.predict_proba(X).round(3))
print("Predictions:", model.predict(X))
print("Final loss:", round(model.loss_history[-1], 4))

Probabilities:
 [[0.943 0.056 0.001]
 [0.935 0.025 0.039]
 [0.979 0.014 0.006]
 [0.005 0.995 0.   ]
 [0.012 0.988 0.001]
 [0.022 0.978 0.   ]
 [0.    0.    1.   ]
 [0.002 0.008 0.99 ]
 [0.    0.002 0.997]]
Predictions: ['A' 'A' 'A' 'B' 'B' 'B' 'C' 'C' 'C']
Final loss: 0.0222


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

model.fit(X, y)

print("Predictions:", model.predict(X))
print("Probabilities:\n", model.predict_proba(X))

Predictions: ['A' 'A' 'A' 'B' 'B' 'B' 'C' 'C' 'C']
Probabilities:
 [[0.81116344 0.12156976 0.0672668 ]
 [0.8084113  0.06916295 0.12242575]
 [0.87377179 0.05495394 0.07127428]
 [0.05376267 0.88966215 0.05657518]
 [0.08085267 0.76377006 0.15537726]
 [0.11136598 0.77335772 0.1152763 ]
 [0.07547059 0.05565465 0.86887476]
 [0.11633271 0.15028013 0.73338716]
 [0.06914984 0.12151538 0.80933478]]


In [3]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)